In [1]:
from pathlib import Path
import numpy as np
import nibabel as nb
from pylibCZIrw import czi as pyczi

CZI_PATH   = Path("/Users/hananeboudlal/Desktop/MIO21100401_Cx_104_112_120_128.czi")  
OUT_DIR    = Path("out"); OUT_DIR.mkdir(exist_ok=True)
CHANNEL    = 0
SCENES     = [0, 1,2,3]       
THICK_UM   = 100.0       
EXPONENTS  = [8, 6, 2]    
MARGIN     = 200          
UM_TO_MM   = 1e-3

In [2]:
import xml.etree.ElementTree as ET
with pyczi.open_czi(str(CZI_PATH)) as doc:
    S = doc.scenes_bounding_rectangle
    n = len(SCENES)                       # nb de scenes qu'on traite
    W = [S[i].w for i in SCENES]          # largeur native (px) par scene choisie
    H = [S[i].h for i in SCENES]          # hauteur native (px)
    r = ET.fromstring(doc.raw_metadata)
    sx = [d.find("Value").text for d in r.iter("Distance") if d.get("Id") == "X"][0]
    s0 = abs(float(sx)) * 1e6 * UM_TO_MM  # taille pixel natif (mm/px)

print("scenes:", n, "| pixel natif:", round(s0*1e3, 5), "um | W:", W, "| H:", H)

scenes: 4 | pixel natif: 0.32486 um | W: [55424, 56964, 58503, 60032] | H: [26332, 27810, 27787, 29371]


In [3]:
def dsize(native, f):
    return round(native / f)              # taille estimee (arrondi a l'entier le plus proche)

def affine(nat_x, nat_y, f, k, n):
    s = s0 * f                            # taille pixel a cette resolution
    ox = -nat_x * s0 / 2                  # origine COMMUNE ne depend pas de facteur ds 
    oy = -nat_y * s0 / 2                  
    oz = (k - (n - 1) / 2) * THICK_UM * UM_TO_MM
    return np.array([[s,0,0,ox],[0,s,0,oy],[0,0,THICK_UM*UM_TO_MM,oz],[0,0,0,1.]])

def save(data, A, name):
    data = np.asarray(data, np.float32)   # datatype unifie
    if data.ndim == 2: data = data[:, :, None]
    img = nb.Nifti1Image(data, A)
    img.set_sform(A, code=1); img.set_qform(A, code=1)
    img.header.set_xyzt_units("mm")
    nb.save(img, str(OUT_DIR / name))
# dimension de la boite englobante des scenes , methode pour avoir une boite qui peut contenir tout un nombre entier de pixels 
# si max width = 60032 + margin=200 = 60232  et si on calcule combien de pixel de taille 256 elle peut contenir on trouve =>
# 60232/256 = 235,28125 ce qui nest pas accepte par le logiciel 
# donc on arrondi dabord => 236 
# et si on fait 236 x 256 = 60416  pixel natifs dans la boite 
Bx = max(W) + MARGIN               
By = max(H) + MARGIN
print("boite (natif):", Bx, "x", By)

boite (natif): 60232 x 29571


In [4]:
sizes = {}                                       
with pyczi.open_czi(str(CZI_PATH)) as doc:
    S = doc.scenes_bounding_rectangle
    for e in EXPONENTS:
        f = 2 ** e
        wDS = [W[k] // f for k in range(n)]     # largeur réduite de chaque coupe a une résolution f donné  
        hDS = [H[k] // f for k in range(n)]     
        wpad = max(wDS) + MARGIN                  
        hpad = max(hDS) + MARGIN
        
        vol = np.zeros((wpad, hpad, n), np.float32)
        ds_sizes = []
        for k, sc in enumerate(SCENES):                     
            a = np.asarray(doc.read(roi=(S[sc].x, S[sc].y, W[k], H[k]),
                           plane={"C": CHANNEL}, scene=sc, zoom=1/f)).squeeze()
            a = np.swapaxes(a, 0, 1).astype(np.float32)    
            wf, hf = a.shape
            ds_sizes.append((wf, hf))

            matrice_coupeDS = affine(W[k], H[k], f, k, n)              
            pbx = round((wpad - wf) / 2)
            pby = round((hpad - hf) / 2)          

            matrice_pad = matrice_coupeDS.copy()
            matrice_pad[:3, 3] = (matrice_coupeDS[:3, 3]
                            - pbx * matrice_coupeDS[:3, 0]            
                            - pby * matrice_coupeDS[:3, 1])            
            # hf => hauteur de la coupe réduite 
            pad = np.pad(a, ((pbx, wpad-wf-pbx), (pby, hpad-hf-pby))) # noir gauche/droite , haut/bas
            vol[:, :, k] = pad

            if k == 0:
                A_vol = matrice_pad.copy()                        

            save(a,   matrice_coupeDS,  f"scene{sc}_res-{e}x_downsampled.nii.gz")
            save(pad, matrice_pad, f"scene{sc}_res-{e}x_padded.nii.gz")

        save(vol, A_vol, f"volume_res-{e}x.nii.gz")         
        sizes[e] = {"padded": (wpad, hpad), "downsampled": ds_sizes}
print("fait ->", OUT_DIR.resolve())

fait -> /Users/hananeboudlal/Desktop/ALTERNANCE-ALL/mimosa/out


In [5]:
print("=== REDUITE : native / facteur (attendu)  vs  ZEN (reel) ===")
for e in EXPONENTS:
    f = 2 ** e
    for k, sc in enumerate(SCENES):
        wf, hf = sizes[e]["downsampled"][k]              # ZEN (reel)
        aw, ah = W[k] / f, H[k] / f                      # attendu = native / facteur (exact)
        print(f"res-{e}x scene{sc}: native {W[k]}x{H[k]}"
              f"  | /{f} = {aw:.2f} x {ah:.2f}"
              f"  | pylibCZI zoom {wf} x {hf}"
              f"  | ecart(-attendu) {wf-aw:+.2f}, {hf-ah:+.2f}")
    print()
print("=== PADDEE : taille par resolution (proportionnalite) ===")
ref = max(EXPONENTS)                          # resolution la plus grossiere = reference
wp_ref, hp_ref = sizes[ref]["padded"]
for e in EXPONENTS:
    fac = 2 ** (ref - e)                       # facteur attendu vs la reference
    wp, hp = sizes[e]["padded"]
    print(f"res-{e}x : paddee {wp} x {hp}"
          f"  | attendu (res-{ref}x x{fac}) = {wp_ref*fac} x {hp_ref*fac}"
          f"  | ecart {wp - wp_ref*fac:+d}, {hp - hp_ref*fac:+d}")
print()

=== REDUITE : native / facteur (attendu)  vs  ZEN (reel) ===
res-8x scene0: native 55424x26332  | /256 = 216.50 x 102.86  | ZEN 216 x 102  | ecart(ZEN-attendu) -0.50, -0.86
res-8x scene1: native 56964x27810  | /256 = 222.52 x 108.63  | ZEN 222 x 108  | ecart(ZEN-attendu) -0.52, -0.63
res-8x scene2: native 58503x27787  | /256 = 228.53 x 108.54  | ZEN 228 x 108  | ecart(ZEN-attendu) -0.53, -0.54
res-8x scene3: native 60032x29371  | /256 = 234.50 x 114.73  | ZEN 234 x 114  | ecart(ZEN-attendu) -0.50, -0.73

res-6x scene0: native 55424x26332  | /64 = 866.00 x 411.44  | ZEN 866 x 411  | ecart(ZEN-attendu) +0.00, -0.44
res-6x scene1: native 56964x27810  | /64 = 890.06 x 434.53  | ZEN 890 x 434  | ecart(ZEN-attendu) -0.06, -0.53
res-6x scene2: native 58503x27787  | /64 = 914.11 x 434.17  | ZEN 914 x 434  | ecart(ZEN-attendu) -0.11, -0.17
res-6x scene3: native 60032x29371  | /64 = 938.00 x 458.92  | ZEN 938 x 458  | ecart(ZEN-attendu) +0.00, -0.92

res-2x scene0: native 55424x26332  | /4 = 138